In [ ]:
import requests
import json
import pandas as pd

URL = 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2020-01-01'

response = requests.get(URL)
data = response.json()


In [81]:
df_earthquakes = pd.json_normalize(data['features'])

In [82]:
print(df_earthquakes.columns)


Index(['type', 'id', 'properties.mag', 'properties.place', 'properties.time',
       'properties.updated', 'properties.tz', 'properties.url',
       'properties.detail', 'properties.felt', 'properties.cdi',
       'properties.mmi', 'properties.alert', 'properties.status',
       'properties.tsunami', 'properties.sig', 'properties.net',
       'properties.code', 'properties.ids', 'properties.sources',
       'properties.types', 'properties.nst', 'properties.dmin',
       'properties.rms', 'properties.gap', 'properties.magType',
       'properties.type', 'properties.title', 'geometry.type',
       'geometry.coordinates'],
      dtype='str')


In [83]:
df_earthquakes = df_earthquakes[[
    'id', 'properties.mag', 'properties.place', 'properties.time',
       'properties.updated', 
       'properties.felt', 'properties.cdi',
       'properties.mmi', 'properties.alert', 
       'properties.tsunami', 'properties.sig','properties.nst', 'properties.dmin',
       'properties.rms', 'properties.gap', 'properties.magType',
       'properties.type', 
       'geometry.coordinates']]

In [84]:
df_earthquakes.rename(columns={
       'id': 'ID', 
       'properties.mag': 'Magnitude', 
       'properties.place': 'Place', 
       'properties.time': 'EventTime',
       'properties.updated': 'UpdateTime',
       'properties.felt': 'Felt', 
       'properties.cdi': 'Cdi',
       'properties.mmi': 'Mmi', 
       'properties.alert': 'Alert',
       'properties.tsunami': 'Tsunami', 
       'properties.sig': 'Sig',
       'properties.nst': 'Nst', 
       'properties.dmin': 'Dmin',
       'properties.rms': 'Rms', 
       'properties.gap': 'Gap', 
       'properties.magType': 'MagType',
       'properties.type': 'Type', 
       'geometry.coordinates': 'coordinates'}, inplace=True)

In [85]:
df_earthquakes['Lon'] = df_earthquakes['coordinates'].str[0]
df_earthquakes['Lat'] = df_earthquakes['coordinates'].str[1]
df_earthquakes['DepthKm'] = df_earthquakes['coordinates'].str[2]

df_earthquakes.drop(columns=['coordinates'], inplace=True)


In [86]:
print(df_earthquakes.info())

<class 'pandas.DataFrame'>
RangeIndex: 10179 entries, 0 to 10178
Data columns (total 20 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          10179 non-null  str    
 1   Magnitude   10179 non-null  float64
 2   Place       10179 non-null  str    
 3   EventTime   10179 non-null  int64  
 4   UpdateTime  10179 non-null  int64  
 5   Felt        457 non-null    float64
 6   Cdi         457 non-null    float64
 7   Mmi         164 non-null    float64
 8   Alert       60 non-null     str    
 9   Tsunami     10179 non-null  int64  
 10  Sig         10179 non-null  int64  
 11  Nst         10178 non-null  float64
 12  Dmin        10174 non-null  float64
 13  Rms         10178 non-null  float64
 14  Gap         10177 non-null  float64
 15  MagType     10179 non-null  str    
 16  Type        10179 non-null  str    
 17  Lon         10179 non-null  float64
 18  Lat         10179 non-null  float64
 19  DepthKm     10179 non-null  float64


In [87]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_url = URL.create(
    "mssql+pyodbc",
    host=r"localhost",   # or just "localhost"
    database="NasaEvents",
    username="felip",  # replace with your username
    password="1990",  # replace with your password
    query={
        "driver": "ODBC Driver 17 for SQL Server",
        "trusted_connection": "yes",      
        "TrustServerCertificate": "yes",
    },
)

engine = create_engine(connection_url)

df_earthquakes.to_sql('bronze_earthquakes', con=engine, if_exists='replace', index=False)

91